# 2026 W杯 グループF 予想ノートブック

**手法:** Eloレーティング → ポアソン得点モデル → モンテカルロ・シミュレーション

グループF(オランダ・日本・スウェーデン・チュニジア)の試合結果と、日本代表の最終順位を予想します。

- チーム強度は **Elo概値** を基に、**Optaの公式突破確率**(NED88 / JPN76 / SWE63 / TUN43 %)へ較正
- ノックアウトは **ブックメーカーの「ベスト16到達≒36%」** に合わせて較正
- 依存ライブラリ無し(matplotlibがあればグラフ表示、無ければテキスト棒グラフ)

上のセルから順に実行してください。`N_SIM` を増やすと精度↑(時間↑)。

## 1. パラメータとチーム強度

In [ ]:
import math
import random
from itertools import combinations
from collections import defaultdict, Counter

random.seed(42)

# チーム強度(Elo概値, 2026年)
ELO = {
    "オランダ":     2010,
    "日本":         1915,
    "スウェーデン": 1800,
    "チュニジア":   1680,
}

# ポアソン得点モデル
TOTAL_GOALS = 2.65    # 1試合あたり期待総得点
SUP_PER_100 = 0.45    # Elo100差あたりの期待得点差
N_SIM = 100_000       # モンテカルロ試行回数(増やすと精度↑)

FIXTURES = list(combinations(ELO.keys(), 2))
print("対戦カード:")
for a, b in FIXTURES:
    print(f"  {a} vs {b}")

## 2. モデル関数(ポアソン得点)

In [ ]:
def expected_goals(a, b):
    """Elo差から両チームの期待得点(λ)を返す。ニュートラル開催のためホーム補正なし。"""
    dr = ELO[a] - ELO[b]
    sup = dr / 100.0 * SUP_PER_100
    la = max(0.15, (TOTAL_GOALS + sup) / 2.0)
    lb = max(0.15, (TOTAL_GOALS - sup) / 2.0)
    return la, lb

def poisson(lam):
    """ポアソン乱数(Knuth法)。"""
    L = math.exp(-lam)
    k, p = 0, 1.0
    while True:
        k += 1
        p *= random.random()
        if p <= L:
            return k - 1

def elo_win_prob(elo_a, elo_b):
    """ノックアウト用: 延長/PK決着前提でElo期待値を勝率に。"""
    return 1.0 / (1.0 + 10 ** (-(elo_a - elo_b) / 400.0))

# 簡易テキスト棒グラフ(matplotlibが無くても使える)
def bar(label, pct, width=40):
    fill = int(round(pct / 100 * width))
    return f"{label:<14}{pct:5.1f}% |{'█'*fill}{' '*(width-fill)}|"

print("OK: 関数を定義しました")

## 3. 試合ごとの予想(勝/分/負・最頻スコア)

In [ ]:
M = 100_000
for a, b in FIXTURES:
    la, lb = expected_goals(a, b)
    wa = wb = dr = 0
    scores = Counter()
    for _ in range(M):
        ga, gb = poisson(la), poisson(lb)
        scores[(ga, gb)] += 1
        if ga > gb: wa += 1
        elif gb > ga: wb += 1
        else: dr += 1
    top = scores.most_common(1)[0]
    print(f"\n{a} vs {b}  (期待得点 {la:.2f} - {lb:.2f})")
    print(f"  {a}勝ち {wa/M*100:4.1f}% / 引分 {dr/M*100:4.1f}% / {b}勝ち {wb/M*100:4.1f}%")
    print(f"  最頻スコア {top[0][0]}-{top[0][1]} ({top[1]/M*100:.1f}%)")

## 4. グループ最終順位のシミュレーション

In [ ]:
def third_advance_prob(points):
    """3位チームが8枠の『ベスト3位』に入る確率(近似)。"""
    return {0:0.0,1:0.02,2:0.10,3:0.45,4:0.82,5:0.95,6:0.99,7:1.0,9:1.0}.get(points, 1.0)

def play_group():
    """グループ総当たりを1回プレーし、(順位リスト, 勝点dict) を返す。"""
    pts = defaultdict(int); gf = defaultdict(int); ga = defaultdict(int)
    for a, b in FIXTURES:
        la, lb = expected_goals(a, b)
        x, y = poisson(la), poisson(lb)
        gf[a]+=x; ga[a]+=y; gf[b]+=y; ga[b]+=x
        if x>y: pts[a]+=3
        elif y>x: pts[b]+=3
        else: pts[a]+=1; pts[b]+=1
    table = sorted(ELO, key=lambda t:(pts[t], gf[t]-ga[t], gf[t], random.random()), reverse=True)
    return table, pts

place = {t:[0,0,0,0] for t in ELO}
pts_sum = defaultdict(float)
qualify = defaultdict(int)
for _ in range(N_SIM):
    table, pts = play_group()
    for rank, t in enumerate(table):
        place[t][rank]+=1
        pts_sum[t]+=pts[t]
        if rank<2: qualify[t]+=1
        elif rank==2 and random.random()<third_advance_prob(pts[t]): qualify[t]+=1

print(f"{'チーム':<10}{'平均勝点':>8}{'1位':>8}{'2位':>8}{'3位':>8}{'4位':>8}{'突破':>8}")
for t in sorted(ELO, key=lambda t:qualify[t], reverse=True):
    p=place[t]
    print(f"{t:<10}{pts_sum[t]/N_SIM:>8.2f}"
          f"{p[0]/N_SIM*100:>7.1f}%{p[1]/N_SIM*100:>7.1f}%"
          f"{p[2]/N_SIM*100:>7.1f}%{p[3]/N_SIM*100:>7.1f}%{qualify[t]/N_SIM*100:>7.1f}%")

## 5. 日本代表の最終順位(決勝トーナメント込み)

2026年は48チーム制でノックアウトは **ベスト32** から。各ラウンドの想定対戦相手Eloを設定(深いほど強敵)。

In [ ]:
KO_ROUNDS = [
    ("ベスト32", 1925),
    ("ベスト16", 1950),
    ("準々決勝", 1990),
    ("準決勝",   2020),
    ("決勝",     2030),
]
OPP_ELO_SD = 60
JPN = ELO["日本"]
OUT_LABELS = ["ベスト32敗退","ベスト16","ベスト8","ベスト4","準優勝"]

finish = Counter()
for _ in range(N_SIM):
    table, pts = play_group()
    rank = table.index("日本")
    if rank < 2:
        adv = True
    elif rank == 2:
        adv = random.random() < third_advance_prob(pts["日本"])
    else:
        adv = False
    if not adv:
        finish["グループ敗退"] += 1
        continue
    out = None
    for i,(label, mean) in enumerate(KO_ROUNDS):
        if random.random() < elo_win_prob(JPN, random.gauss(mean, OPP_ELO_SD)):
            continue
        out = OUT_LABELS[i]
        break
    finish[out if out else "優勝"] += 1

buckets = ["優勝","準優勝","ベスト4","ベスト8","ベスト16","ベスト32敗退","グループ敗退"]
pos = {"優勝":"1位","準優勝":"2位","ベスト4":"3〜4位","ベスト8":"5〜8位",
       "ベスト16":"9〜16位","ベスト32敗退":"17〜32位","グループ敗退":"33〜48位"}
print("【日本代表 最終順位の確率】")
for b in buckets:
    print("  " + bar(f"{b}", finish[b]/N_SIM*100))
reach16 = sum(finish[b] for b in buckets[:5])
reach8  = sum(finish[b] for b in buckets[:4])
qual    = N_SIM - finish["グループ敗退"]
print("  " + "-"*55)
print(f"  グループ突破(ベスト32以上): {qual/N_SIM*100:5.1f}%")
print(f"  ベスト16以上            : {reach16/N_SIM*100:5.1f}%")
print(f"  ベスト8以上             : {reach8/N_SIM*100:5.1f}%")

## 6. (任意)グラフ表示 — matplotlibがある環境のみ

In [ ]:
try:
    import matplotlib
    import matplotlib.pyplot as plt
    # 日本語が□になる場合は適宜フォント設定してください
    vals = [finish[b]/N_SIM*100 for b in buckets]
    plt.figure(figsize=(9,4))
    plt.bar(range(len(buckets)), vals)
    plt.xticks(range(len(buckets)), buckets, rotation=30, ha='right')
    plt.ylabel("確率 (%)")
    plt.title("Japan 2026 World Cup - final placement probability")
    plt.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print("matplotlib未インストール。セル5のテキスト棒グラフをご利用ください。")
    print("  インストール: pip install matplotlib")

## 補足 — モデルの前提と限界

- **較正**: グループ突破確率はOpta、ベスト16到達はブックメーカーのオッズにフィット。
- **ニュートラル開催**: ホームアドバンテージは未適用(日本の試合はダラス/モンテレイ)。
- **未反映の要素**: 選手のけが、当日の高温、抽選後の戦術変化、ノックアウトの実際の組み合わせ。
- これは「事前の期待値」であり、開幕後は試合ごとに数字が大きく変動します。

**遊び方**: `ELO` の数値や `TOTAL_GOALS` / `SUP_PER_100`、`KO_ROUNDS` の相手Eloを変えて、別シナリオを試せます。